In [ ]:
# ── STEP 1.1.0: Initialize 1.1 Environment ──────────
!pip install biopython peptides

In [ ]:
# ── STEP 1.1: Load existing complex and establish baseline ──────────
from Bio.SeqUtils.ProtParam import ProteinAnalysis

COMPLEX_PDB = "/content/lead_complex.pdb"     # protein-peptide docked complex
SOLO_PEPTIDE_PDB = "/content/lead_peptide.pdb" # solo peptide structure
LEAD_SEQUENCE = "HIYTHMSHFIKQCFSLP"

# Baseline metrics
# Cleavage counts are logged manually via PeptideCutter, if total cleavage count exceeds
BASELINE = {
    "sequence":    LEAD_SEQUENCE,
    "moe_score":   -41.71,
    "gravy":       None,
    "trypsin_cuts": 1,
    "pepsin_cuts": 5,
    "chymotrypsin_cuts": 3
}

# Compute GRAVY from solo peptide sequence
analysis = ProteinAnalysis(LEAD_SEQUENCE)
BASELINE["gravy"] = round(analysis.gravy(), 3)

print("=== BASELINE (from existing complex + solo peptide) ===")
for k, v in BASELINE.items():
    print(f"  {k}: {v}")

# Derived thresholds
GRAVY_HARD_CAP           = BASELINE["gravy"]
MOE_DISCARD_THRESHOLD    = BASELINE["moe_score"] * 0.75

In [ ]:
# ── STEP 1.2.0: Initialize 1.2 Environment ──────────
import os
import shutil

# Clone the repository
!git clone https://github.com/dauparas/ProteinMPNN.git
%cd /content/ProteinMPNN

# Create directories for inputs and outputs
os.makedirs("my_inputs", exist_ok=True)
os.makedirs("my_outputs", exist_ok=True)

# Move the relaxed complex from the root to the inputs folder
source_pdb = "/content/lead_complex.pdb"
destination_pdb = "/content/ProteinMPNN/my_inputs/lead_complex.pdb"

if os.path.exists(source_pdb):
    shutil.copy(source_pdb, destination_pdb)
    print(f"[*] Successfully copied lead complex PDB to: {destination_pdb}")
else:
    print(f"[-] Error: Could not find {source_pdb}. Did PyRosetta finish saving it?")

print("[+] Workspace ready. Current directory:", os.getcwd())

In [ ]:
# ProteinMPNN Wrapper
%%writefile design_peptide.py
import argparse
import json
import subprocess
from pathlib import Path

def build_mpnn_dicts(pdb_name, chain_id, args, out_dir):
    """
    Translates custom design logic arguments into the fixed_positions,
    omit_AA, and bias_AA JSONL dictionaries required by ProteinMPNN.
    """
    standard_alphabet = set("ACDEFGHIKLMNPQRSTVWY")

    # 1. Handle --freeze
    fixed_dict = {pdb_name: {chain_id: [int(x) for x in args.freeze]}}
    fixed_json_path = out_dir / f"{pdb_name}_fixed.jsonl"
    with open(fixed_json_path, 'w') as f:
        json.dump(fixed_dict, f)
        f.write('\n')

    # 2. Handle --mutate_conditional and --restrict
    per_position_omit = {}

    def process_conditional(arg_list):
        for item in arg_list:
            parts = item.split(':')
            if len(parts) != 2:
                print(f"[-] Warning: Invalid format '{item}'. Expected pos:A,B,C")
                continue
            pos = int(parts[0].strip())
            allowed_aas = set(parts[1].upper().replace(',', ''))
            omitted_aas = standard_alphabet - allowed_aas
            if pos not in per_position_omit:
                per_position_omit[pos] = set()
            per_position_omit[pos].update(omitted_aas)

    if args.mutate_conditional:
        process_conditional(args.mutate_conditional)

    if args.restrict:
        process_conditional(args.restrict)

    omit_json_path = None
    if per_position_omit:
        omit_string_to_positions = {}
        for pos, aas in per_position_omit.items():
            omit_str = "".join(sorted(aas))
            if omit_str not in omit_string_to_positions:
                omit_string_to_positions[omit_str] = []
            omit_string_to_positions[omit_str].append(pos)

        omit_entries = []
        for omit_str, positions in omit_string_to_positions.items():
            omit_entries.append([sorted(positions), omit_str])

        omit_dict = {pdb_name: {chain_id: omit_entries}}
        omit_json_path = out_dir / f"{pdb_name}_omit.jsonl"
        with open(omit_json_path, 'w') as f:
            json.dump(omit_dict, f)
            f.write('\n')

    # 3. Handle --bias (Global Amino Acid Weighting)
    bias_json_path = None
    if args.bias:
        bias_dict_inner = {}
        for item in args.bias:
            parts = item.split(':')
            if len(parts) == 2:
                aa = parts[0].strip().upper()
                score = float(parts[1].strip())
                bias_dict_inner[aa] = score

        if bias_dict_inner:
            bias_dict = {pdb_name: {chain_id: bias_dict_inner}}
            bias_json_path = out_dir / f"{pdb_name}_bias.jsonl"
            with open(bias_json_path, 'w') as f:
                json.dump(bias_dict, f)
                f.write('\n')

    return fixed_json_path, omit_json_path, bias_json_path


def filter_fasta_by_net_charge(fasta_path, target_charges):
    if not fasta_path.exists() or not target_charges:
        return

    with open(fasta_path, 'r') as f:
        content = f.read()

    entries = content.strip().split(">")
    if len(entries) < 2:
        return

    filtered_entries = [f">{entries[1].strip()}"]

    for entry in entries[2:]:
        lines = entry.strip().split("\n")
        if not lines:
            continue
        sequence = "".join(lines[1:]).strip().upper()

        k_count = sequence.count('K')
        r_count = sequence.count('R')
        d_count = sequence.count('D')
        e_count = sequence.count('E')
        net_charge = (k_count + r_count) - (d_count + e_count)

        if net_charge in target_charges:
            filtered_entries.append(f">{entry.strip()}")

    if len(filtered_entries) > 1:
        with open(fasta_path, 'w') as f:
            f.write("\n".join(filtered_entries) + "\n")
        print(f"[+] Charge Filter: Retained {len(filtered_entries)-1} generated variants "
              f"matching net charges {target_charges}.")
    else:
        print("[-] Warning: No generated sequences met the charge criteria.")


def main():
    parser = argparse.ArgumentParser(description="Custom ProteinMPNN Peptide Designer")

    parser.add_argument("--pdb", type=str, required=True, help="Input PDB template file")
    parser.add_argument("--chain", type=str, default="B", help="Peptide chain to design (usually B)")
    parser.add_argument("--out_folder", type=str, default="mpnn_outputs", help="Output directory")
    parser.add_argument("--num_seqs", type=int, default=10, help="Number of sequences to generate")
    parser.add_argument("--temp", type=float, default=0.1, help="Sampling temperature")

    parser.add_argument("--freeze", type=int, nargs='+', default=[], help="Residue indices to lock")
    parser.add_argument("--mutate", type=int, nargs='+', default=[], help="Residue indices to freely mutate")
    parser.add_argument("--mutate_conditional", type=str, nargs='+', default=[], help="Format POS:A,B,C")
    parser.add_argument("--restrict", type=str, nargs='+', default=[], help="Format POS:A,B,C")
    parser.add_argument("--required_charge", type=str, default="", help="Allowed net charges (e.g., -1,0,1)")

    # NEW: Bias argument
    parser.add_argument("--bias", type=str, nargs='+', default=[],
                        help="Format AA:Score. Negative values penalize. (e.g., --bias L:-1.5 E:1.0)")

    args = parser.parse_args()

    pdb_path = Path(args.pdb).resolve()
    pdb_name = pdb_path.stem
    out_dir = Path(args.out_folder).resolve()
    out_dir.mkdir(parents=True, exist_ok=True)
    mpnn_script = Path("protein_mpnn_run.py").resolve()

    print("\n" + "="*60)
    print(f"[*] Processing ProteinMPNN Design Run for: {pdb_name}")
    print("="*60)

    fixed_json, omit_json, bias_json = build_mpnn_dicts(pdb_name, args.chain, args, out_dir)

    cmd = [
        "python", str(mpnn_script),
        "--pdb_path", str(pdb_path),
        "--pdb_path_chains", args.chain,
        "--out_folder", str(out_dir),
        "--num_seq_per_target", str(args.num_seqs),
        "--sampling_temp", str(args.temp),
        "--batch_size", "100",
        "--fixed_positions_jsonl", str(fixed_json)
    ]

    if omit_json:
        cmd.extend(["--omit_AA_jsonl", str(omit_json)])
    if bias_json:
        cmd.extend(["--bias_AA_jsonl", str(bias_json)])

    print(f"[*] Executing ProteinMPNN Model...")
    try:
        subprocess.run(cmd, check=True)

        fasta_output = out_dir / "seqs" / f"{pdb_name}.fa"

        if fasta_output.exists() and args.required_charge:
            print("\n[*] Applying Net Charge Gateway Filter...")
            target_charges = [int(c.strip()) for c in args.required_charge.split(',')]
            filter_fasta_by_net_charge(fasta_output, target_charges)

        print("\n[+] Design Pipeline Complete!")
        if fasta_output.exists():
            print(f"[+] Output FASTA saved to: {fasta_output}")

    except subprocess.CalledProcessError as e:
        print(f"\n[-] Error running ProteinMPNN: {e}")

if __name__ == "__main__":
    main()

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!rm -rf my_outputs && mkdir -p my_outputs/seqs
!python design_peptide.py \
    --pdb my_inputs/lead_complex.pdb \
    --chain A \
    --freeze 2 5 8 9 12 15 16 \
    --num_seqs 5000 \
    --temp 0.5 \
    --bias L:0.5 I:0.5 A:1.0 S:0.5 H:1.0 F:1.0 K:1.0 D:-1.0 E:-1.0 \
    --mutate_conditional 1:S,L,I,A 3:A,S,G,I 4:A,S,I,V 6:H,I,S 7:A,S,I,H 10:H,I,K,S 11:H,I,K,S 13:S,A,I \
    --restrict 14:K,R,F,Y,W,L \
    --required_charge 1,2 \
    --out_folder my_outputs

In [ ]:
# ── STEP 1.3.0: Prepare for Biophysical Triage ──────────
%cd /content/ProteinMPNN
import re
import os

OUTPUT_DIR = "my_outputs"

def parse_mpnn_fasta(fasta_path):
    """Parse ProteinMPNN output FASTA. Returns list of (name, sequence) tuples."""
    sequences = []
    current_name = None
    with open(fasta_path) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                current_name = line[1:]
            elif line and current_name:
                # Skip the original sequence (score=0.0000 is the reference)
                if "score" in current_name and current_name != "original":
                    sequences.append((current_name, line))
    return sequences

# Find the output FASTA
fasta_files = [f for f in os.listdir(f"{OUTPUT_DIR}/seqs/") if f.endswith(".fa")]
print(f"Found FASTA files: {fasta_files}")

all_candidates = []
for fa in fasta_files:
    seqs = parse_mpnn_fasta(f"{OUTPUT_DIR}/seqs/{fa}")
    all_candidates.extend(seqs)

# Deduplicate
unique_seqs = list({seq: name for name, seq in all_candidates}.items())
print(f"Total unique candidates generated: {len(unique_seqs)}")

In [ ]:
# ── STEP 1.3: Biophysical Triage ──────────
import os
from Bio.SeqUtils.ProtParam import ProteinAnalysis

OUTPUT_DIR = "my_outputs"

# 1. Parse all generated sequences
fasta_files = [f for f in os.listdir(f"{OUTPUT_DIR}/seqs/") if f.endswith(".fa")]
all_candidates = []
for fa in fasta_files:
    all_candidates.extend(parse_mpnn_fasta(f"{OUTPUT_DIR}/seqs/{fa}"))

unique_seqs = {seq: name for name, seq in all_candidates}

# 2. Apply Solubility Filter
passed_candidates = {}
for seq, name in unique_seqs.items():
    try:
        gravy = round(ProteinAnalysis(seq).gravy(), 3)
        if gravy <= GRAVY_HARD_CAP:
            passed_candidates[seq] = {"name": name, "gravy": gravy}
    except Exception:
        continue

print(f"Candidates passing GRAVY: {len(passed_candidates)}")

# 3. Export for PeptideCutter
sorted_candidates = sorted(passed_candidates.items(), key=lambda x: x[1]['gravy'])

with open("peptide_cutter_batch.fasta", "w") as f:
    for i, (seq, data) in enumerate(sorted_candidates):
        f.write(f">Candidate_{i+1:03d}_GRAVY_{data['gravy']}\n{seq}\n")

print("\n[!] Export complete. Run 'peptide_cutter_batch.fasta' through PeptideCutter.")

In [ ]:
cleavage_cleared_sequences = [
    "SISSHSSHFSSQSKSLK",
    "SISAHSAHFKSQSRSLD",
    "AISAHSSHFKHQAKSLE",
    "SISAHSAHFSKQSRSLP",
    "AISVHSSHFKSQSRSLE",
    "SIAAHHSHFSKQAKSLY",
    "IIASHHAHFKKQAKSLD",
    "SISAHSSHFSSQAKSLK",
    "IIGIHSHHFSKQSRSLE",
    "SISIHSSHFHKQAKSLS",
    "LISAHSSHFKSQSRSLP",
    "LIAAHHAHFHKQSKSLP",
    "LISAHHAHFKSQSKSLY",
    "SIAAHSHHFSSQARSLP",
    "LIGSHSHHFHSQSRSLL",
    "LIASHHSHFSKQAKSLS",
    "AISAHSSHFSSQSKSLP",
    "IIASHSAHFKSQSKSLE",
    "IIASHSSHFKSQAKSLN",
    "SIASHSAHFKHQAKSLM",
    "SISSHIHHFKKQAFSLP",
    "AIAAHHAHFSSQSKSLN",
    "AIASHSAHFSKQARSLP",
    "LISSHSAHFSKQAKSLP",
    "AIAAHHAHFKSQARSLP",
    "IISSHHSHFKKQALSLP",
    "AIASHHAHFSKQIRSLP",
    "LIAAHHAHFKHQAKSLP",
    "SISSHSSHFSKQALSLP",
    "AIIAHSSHFSKQSKSLP",
    "LIAAHSSHFHSQSKSLP",
    "AIAVHHSHFIKQSRSLE",
    "SIAAHSSHFKIQARSLD",
    "AIASHSIHFSKQARSLD",
    "SIISHSSHFKHQSLSLP",
    "LISAHHAHFSSQSRSLS",
    "LISIHHAHFSKQAKSLE",
    "LISAHHSHFIKQAKSLD",
    "SISAHSAHFKIQAKSLD",
    "SIASHHAHFSIQSKSLP",
    "AIAAHSSHFKSQSKSLA",
    "SIAVHHAHFKIQSKSLD",
    "SIAAHISHFKSQSKSLT",
    "AIISHIAHFHKQSKSLE",
    "SIGSHSAHFIKQAKSLS",
    "SISAHSSHFSIQSKSLS",
    "IIIAHHSHFHKQAKSLS",
    "SIAAHSSHFHSQAKSLA",
    "SISSHIAHFHSQAKSLS",
    "AISAHHAHFKKQALSLP",
    "SIAIHHSHFKKQALSLP"
]


print(f"\nSequences cleared for Notebook 1.4 (Immunogenicity): {len(cleavage_cleared_sequences)}")

In [ ]:
# ── STEP 1.4.0: Initialize environment for Immunogenicity Screening ──────────
!pip install -q mhcnuggets pandas
!apt-get install -y -qq tcsh gawk

In [ ]:
# ── STEP 1.4: IMMUNOGENICITY SCREENING (MHCnuggets — fully local, IC50-based) ──────────
import os
import sys
import warnings
import tempfile
import tensorflow as tf
import pandas as pd
from pathlib import Path
from contextlib import redirect_stdout, redirect_stderr
from keras.optimizers import Adam as _OriginalAdam

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')

# ── Patch MHCnuggets' Keras 2 Adam(lr=...) call for Keras 3 compatibility ─────
class _PatchedAdam(_OriginalAdam):
    def __init__(self, lr=None, learning_rate=0.001, **kwargs):
        if lr is not None:
            learning_rate = lr
        super().__init__(learning_rate=learning_rate, **kwargs)

import mhcnuggets.src.predict as _mp
_mp.Adam = _PatchedAdam
from mhcnuggets.src.predict import predict as _mhcnuggets_predict

IC50_THRESHOLD = 50  # nM — strong binders only

MHC_I_ALLELES  = ["HLA-A02:01", "HLA-A01:01", "HLA-B07:02", "HLA-B44:02"]
MHC_II_ALLELES = ["HLA-DRB101:01", "HLA-DRB103:01", "HLA-DRB104:01",
                  "HLA-DRB107:01", "HLA-DRB111:01", "HLA-DRB115:01"]


def _silent_predict(**kwargs):
    """Run mhcnuggets predict() with all stdout/stderr/warnings suppressed."""
    with open(os.devnull, 'w') as devnull, \
         redirect_stdout(devnull), \
         redirect_stderr(devnull), \
         warnings.catch_warnings():
        warnings.simplefilter("ignore")
        _mhcnuggets_predict(**kwargs)


def _scan(sequence, mhc_class, alleles, peptide_length):
    liabilities = []
    sub_peptides = [sequence[i:i+peptide_length]
                    for i in range(len(sequence) - peptide_length + 1)]
    if not sub_peptides:
        return liabilities

    with tempfile.NamedTemporaryFile(mode='w', suffix='.peps', delete=False) as f:
        f.write('\n'.join(sub_peptides))
        in_path = f.name
    out_path = in_path + '_out.csv'

    for allele in alleles:
        try:
            _silent_predict(
                class_=mhc_class,
                peptides_path=in_path,
                mhc=allele,
                output=out_path
            )
            results = pd.read_csv(out_path)
            label = "MHC-I" if mhc_class == 'I' else "MHC-II"
            for _, row in results[results['ic50'] < IC50_THRESHOLD].iterrows():
                liabilities.append(
                    f"{label} [{allele}] {row['peptide']} (IC50={row['ic50']:.1f} nM)"
                )
        except Exception as e:
            print(f"  [!] MHCnuggets error ({mhc_class} / {allele}): {e}")

    os.unlink(in_path)
    if os.path.exists(out_path):
        os.unlink(out_path)

    return liabilities


def scan_mhc1(sequence):
    return _scan(sequence, 'I',  MHC_I_ALLELES,  peptide_length=9)

def scan_mhc2(sequence):
    return _scan(sequence, 'II', MHC_II_ALLELES, peptide_length=15)


# ── Run screening ──────────────────────────────────────────────────────────────
safe_candidates_for_colabfold = []

print(f"Executing sliding-window MHC Class I & II scan on "
      f"{len(cleavage_cleared_sequences)} peptides...\n")

for seq in cleavage_cleared_sequences:
    mhc1_hits = scan_mhc1(seq)
    mhc2_hits = scan_mhc2(seq)
    liabilities = mhc1_hits + mhc2_hits

    mhc1_alleles_hit = set(h.split('[')[1].split(']')[0] for h in mhc1_hits)
    mhc2_alleles_hit = set(h.split('[')[1].split(']')[0] for h in mhc2_hits)
    all_alleles_hit  = mhc1_alleles_hit | mhc2_alleles_hit

    if len(all_alleles_hit) >= 2:
        classes_hit = []
        if mhc1_hits: classes_hit.append("MHC-I")
        if mhc2_hits: classes_hit.append("MHC-II")
        print(f"FAILED: {seq} is immunogenic via {' & '.join(classes_hit)}. "
              f"Found {len(liabilities)} MHC-binding motifs across "
              f"{len(all_alleles_hit)} alleles.")
        for hit in liabilities:
            print(f"  → {hit}")
    elif liabilities:
        print(f"WARNED: {seq} has {len(liabilities)} single-allele motif(s) "
              f"(IC50 < {IC50_THRESHOLD} nM), proceeding with caution.")
        safe_candidates_for_colabfold.append(seq)
    else:
        print(f"PASSED: {seq} has IC50 > {IC50_THRESHOLD} nM across all alleles.")
        safe_candidates_for_colabfold.append(seq)

output_path = Path(OUTPUT_DIR) / "final_candidates_for_colabfold.txt"
output_path.write_text("\n".join(safe_candidates_for_colabfold))

print(f"\nScreening Complete: {len(safe_candidates_for_colabfold)} peptides "
      f"proceeding to Phase 3 (ColabFold).")
print(f"[+] Saved to: {output_path}")